In [1]:
# Standard imports
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# BioPython for PDB parsing & superposition
from Bio import PDB

# sklearn for modelling
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score


In [2]:
mmparser = PDB.MMCIFParser(QUIET=True)
sup = PDB.Superimposer()

def load_structure(path):
    """Load a CIF structure file into a Bio.PDB structure object."""
    return mmparser.get_structure(os.path.basename(path), path)


In [3]:
def get_residue(structure, chain_id, resnum):
    """Return the residue with number resnum on chain chain_id."""
    for model in structure:
        if chain_id not in [ch.id for ch in model]:
            continue
        chain = model[chain_id]
        for res in chain:
            if res.get_id()[1] == resnum:   # (hetero flag, resnum, insertion code)
                return res
    return None

def sidechain_atom_names(residue):
    """Side-chain atoms excluding backbone."""
    backbone = {"N", "CA", "C", "O", "OXT"}
    return [atom.get_name() for atom in residue if atom.get_name() not in backbone]

def atoms_coords_for_residue(residue, atom_names=None):
    """Return Nx3 coords of selected atoms; if None → side-chain."""
    if atom_names is None:
        atom_names = sidechain_atom_names(residue)
    coords = []
    for name in atom_names:
        if name in residue:
            coords.append(residue[name].get_coord())
    return np.array(coords)

def all_ca_coords(structure):
    """Return all CA coordinates + (chain,resnum) references."""
    coords = []
    refs = []
    for model in structure:
        for chain in model:
            for res in chain:
                if "CA" in res:
                    coords.append(res['CA'].get_coord())
                    refs.append((chain.id, res.get_id()[1]))
    return np.array(coords), refs


In [7]:
def compute_sidechain_rmsd(wt_path, mut_path, chain_id, resnum):
    """
    Compute side-chain RMSD after superposing WT and mutant CIF structures on CA atoms.
    """
    wt = load_structure(wt_path)
    mut = load_structure(mut_path)

    # ---- Get WT and Mut CA atoms with matching (chain,resnum) keys ----
    def get_ca_atoms(struct):
        ca_dict = {}   # {(chain,resnum): Atom}
        for model in struct:
            for chain in model:
                for res in chain:
                    if "CA" in res:
                        key = (chain.id, res.get_id()[1])
                        ca_dict[key] = res["CA"]
        return ca_dict

    wt_ca_dict = get_ca_atoms(wt)
    mut_ca_dict = get_ca_atoms(mut)

    # Intersection of available CA atoms
    common_keys = list(set(wt_ca_dict.keys()).intersection(set(mut_ca_dict.keys())))
    if len(common_keys) < 3:
        raise RuntimeError(f"Not enough CA atoms in common to superpose (found {len(common_keys)}).")

    # Sort for consistent order
    common_keys.sort()

    wt_atoms = [wt_ca_dict[k] for k in common_keys]
    mut_atoms = [mut_ca_dict[k] for k in common_keys]

    # ---- Superpose mutant onto WT ----
    sup = PDB.Superimposer()
    sup.set_atoms(wt_atoms, mut_atoms)
    sup.apply(mut.get_atoms())

    # ---- Extract mutated residue ----
    wt_res = get_residue(wt, chain_id, resnum)
    mut_res = get_residue(mut, chain_id, resnum)

    if wt_res is None or mut_res is None:
        raise RuntimeError(f"Residue {resnum} not found on chain {chain_id} in one of the structures.")

    # ---- Get matching side-chain atoms ----
    backbone = {"N", "CA", "C", "O", "OXT"}
    wt_sc_names = [a.get_name() for a in wt_res if a.get_name() not in backbone]
    mut_sc_names = [a.get_name() for a in mut_res if a.get_name() not in backbone]

    common_sc = [name for name in wt_sc_names if name in mut_sc_names]

    if len(common_sc) == 0:
        return np.nan, 0

    wt_sc = np.vstack([wt_res[name].get_coord() for name in common_sc])
    mut_sc = np.vstack([mut_res[name].get_coord() for name in common_sc])

    # ---- Compute RMSD ----
    diff = wt_sc - mut_sc
    rmsd = np.sqrt((diff * diff).sum() / len(common_sc))

    return float(rmsd), len(common_sc)


In [8]:
pairs = [
    ("C:\Protein Modelling GitHub Project\data\e.coli36099ef-g.cif", "C:\Protein Modelling GitHub Project\data\ef-ge.colimg165510xmic.cif", "A", 593, "Phe593Leu", "MG1655", "IV"),
    ("C:\Protein Modelling GitHub Project\data\e.coli36099ef-g.cif", "C:\Protein Modelling GitHub Project\data\e.coli3609910xmicef-g.cif", "A", 659, "Pro659Leu", "36099", "V"),
]

rows = []
for wt_path, mut_path, chain, res, name, strain, domain in pairs:
    sc_rmsd, n_atoms = compute_sidechain_rmsd(wt_path, mut_path, chain, res)
    rows.append({
        "mutation": name,
        "position": res,
        "chain": chain,
        "domain": domain,
        "strain": strain,
        "sidechain_rmsd": sc_rmsd,
        "n_sc_atoms": n_atoms,
        "wt_path": wt_path,
        "mut_path": mut_path
    })

df_sc = pd.DataFrame(rows)
df_sc


<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
C:\Users\razon\AppData\Local\Temp\ipykernel_15880\2251758073.py:2: SyntaxWarning: invalid escape sequence '\P'
  ("C:\Protein Modelling GitHub Project\data\e.coli36099ef-g.cif", "C:\Protein Modelling GitHub Project\data\ef-ge.colimg165510xmic.cif", "A", 593, "Phe593Leu", "MG1655", "IV"),
C:\Users\razon\AppData\Local\Temp\ipykernel_15880\2251758073.py:2: SyntaxWarning: invalid escape sequence '\P'
  ("C:\Protein Modelling GitHub Project\data\e.coli36099ef-g.cif", "C:\Protein Modelling GitHub Project\data\ef-ge.colimg165510xmic.cif", "A", 593, "Phe593Leu", "MG1655", "IV"),
C:\Users\razon\AppData

,mutation,position,chain,domain,strain,sidechain_rmsd,n_sc_atoms,wt_path,mut_path
0,Phe593Leu,593,A,IV,MG1655,1.766386,4,C:\Protein Modelling GitHub Project\data\e.col...,C:\Protein Modelling GitHub Project\data\ef-ge...
1,Pro659Leu,659,A,V,36099,0.744506,2,C:\Protein Modelling GitHub Project\data\e.col...,C:\Protein Modelling GitHub Project\data\e.col...


In [9]:
def compute_backbone_rmsd(wt_path, mut_path, chain_id, resnum):
    """
    Compute backbone RMSD (N, CA, C atoms) at a mutated residue,
    after superposing WT and mutant structures on CA atoms.
    """
    wt = load_structure(wt_path)
    mut = load_structure(mut_path)

    # ---- Collect CA atoms for superposition ----
    def get_ca_atoms(struct):
        ca_dict = {}
        for model in struct:
            for chain in model:
                for res in chain:
                    if "CA" in res:
                        key = (chain.id, res.get_id()[1])
                        ca_dict[key] = res["CA"]
        return ca_dict

    wt_ca = get_ca_atoms(wt)
    mut_ca = get_ca_atoms(mut)

    common = sorted(set(wt_ca.keys()).intersection(set(mut_ca.keys())))

    if len(common) < 3:
        raise RuntimeError(f"Not enough CA atoms in common to superpose (found {len(common)}).")

    wt_ca_atoms = [wt_ca[k] for k in common]
    mut_ca_atoms = [mut_ca[k] for k in common]

    # ---- Superpose ----
    sup = PDB.Superimposer()
    sup.set_atoms(wt_ca_atoms, mut_ca_atoms)
    sup.apply(mut.get_atoms())

    # ---- Extract backbone atoms of the mutated residue ----
    backbone_atoms = ["N", "CA", "C"]

    wt_res = get_residue(wt, chain_id, resnum)
    mut_res = get_residue(mut, chain_id, resnum)

    if wt_res is None or mut_res is None:
        raise RuntimeError(f"Residue {resnum} not found on chain {chain_id}.")

    common_bb = [a for a in backbone_atoms if a in wt_res and a in mut_res]
    if len(common_bb) == 0:
        return np.nan, 0

    wt_coords = np.vstack([wt_res[a].get_coord() for a in common_bb])
    mut_coords = np.vstack([mut_res[a].get_coord() for a in common_bb])

    diff = wt_coords - mut_coords
    rmsd = np.sqrt((diff * diff).sum() / len(common_bb))

    return float(rmsd), len(common_bb)


In [10]:
rows = []
for wt_path, mut_path, chain, res, name, strain, domain in pairs:

    # compute side-chain RMSD
    sc_rmsd, sc_atoms = compute_sidechain_rmsd(wt_path, mut_path, chain, res)

    # compute backbone RMSD
    bb_rmsd, bb_atoms = compute_backbone_rmsd(wt_path, mut_path, chain, res)

    rows.append({
        "mutation": name,
        "position": res,
        "chain": chain,
        "domain": domain,
        "strain": strain,
        "sidechain_rmsd": sc_rmsd,
        "n_sc_atoms": sc_atoms,
        "backbone_rmsd": bb_rmsd,
        "n_bb_atoms": bb_atoms,
        "wt_path": wt_path,
        "mut_path": mut_path
    })

df_struct = pd.DataFrame(rows)
df_struct


,mutation,position,chain,domain,strain,sidechain_rmsd,n_sc_atoms,backbone_rmsd,n_bb_atoms,wt_path,mut_path
0,Phe593Leu,593,A,IV,MG1655,1.766386,4,0.733401,3,C:\Protein Modelling GitHub Project\data\e.col...,C:\Protein Modelling GitHub Project\data\ef-ge...
1,Pro659Leu,659,A,V,36099,0.744506,2,0.592988,3,C:\Protein Modelling GitHub Project\data\e.col...,C:\Protein Modelling GitHub Project\data\e.col...
